In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
print(PROJECT_ROOT)

/Users/florianb/Downloads/ai-customer-insights-engine


In [ ]:
import json

import pandas as pd
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from config import config

In [3]:
import importlib

importlib.reload(config)

<module 'src.config.config' from '/Users/florianb/Downloads/ai-customer-insights-engine/src/config/config.py'>

In [4]:
print(f"PROCESSED_DATA_PATH = {config.PROCESSED_DATA_PATH}")

PROCESSED_DATA_PATH = /Users/florianb/Downloads/ai-customer-insights-engine/data/processed/processed_reviews.parquet


In [5]:
df = pd.read_parquet(config.PROCESSED_DATA_PATH)

In [6]:
df.head()

,publication_date,rating,title,review,experience_date,bank,review_id,year,month,year_month
0,2025-03-07 14:48:17+01:00,5,Banque pas chère,Banque pas chère,2025-03-07,boursobank,0,2025,3,2025-03
1,2025-03-07 14:45:35+01:00,5,Modif plafond retrait,"Gestion aisée de mon compte. Pas de frais, auc...",2025-03-07,boursobank,1,2025,3,2025-03
2,2025-03-07 14:35:24+01:00,5,Facilité,Rajouter un bénéficiaire et tout s'est bien dé...,2025-03-07,boursobank,2,2025,3,2025-03
3,2025-03-07 14:20:03+01:00,4,Trop d'étape de sécurité,J'ai mis 4 étoiles parce que je trouve qu'il a...,2025-03-01,boursobank,3,2025,3,2025-03
4,2025-03-07 13:28:32+01:00,4,Je recommande cette carte pour les…,Je recommande cette carte pour les voyageurs e...,2025-03-06,boursobank,4,2025,3,2025-03


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48389 entries, 0 to 48388
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype                       
---  ------            --------------  -----                       
 0   publication_date  48389 non-null  datetime64[us, Europe/Paris]
 1   rating            48389 non-null  int64                       
 2   title             48389 non-null  str                         
 3   review            48389 non-null  str                         
 4   experience_date   48389 non-null  datetime64[us]              
 5   bank              48389 non-null  str                         
 6   review_id         48389 non-null  int64                       
 7   year              48389 non-null  int32                       
 8   month             48389 non-null  int32                       
 9   year_month        48389 non-null  str                         
dtypes: datetime64[us, Europe/Paris](1), datetime64[us](1), int32(2), int64(2), str(4)

In [94]:
sample_reviews = df.sample(
    n=50,
    random_state=32
).reset_index(drop=True)

In [95]:
sample_reviews.head()

,publication_date,rating,title,review,experience_date,bank,review_id,year,month,year_month
0,2023-01-20 16:32:03+01:00,5,Avis sur fortunéo,Très bien pour l'instant à voir dans le temps ...,2023-01-19,fortuneo,18014,2023,1,2023-01
1,2024-03-21 13:33:41+01:00,1,6 Mois que fortuneo me fais tourner en…,"6 Mois que fortuneo me fais tourner en rond , ...",2023-10-23,fortuneo,41170,2024,3,2024-03
2,2023-03-23 11:59:33+01:00,5,Banque en ligne efficace et gratuite,Banque en ligne qui répond parfaitement à mes ...,2023-03-22,fortuneo,10880,2023,3,2023-03
3,2023-05-04 20:46:28+02:00,1,Service client,Délai d'attente téléphonique très long pour un...,2023-01-12,fortuneo,10113,2023,5,2023-05
4,2024-04-02 20:32:22+02:00,5,Ouverture de compte,Tous les problèmes d'ouverture de compte ont é...,2024-02-02,fortuneo,41085,2024,4,2024-04


In [96]:
reviews_text = "\n\n".join(sample_reviews["review"])

In [97]:
llm = ChatOpenAI(
    api_key=config.OPENAI_API_KEY,
    model="gpt-4o-mini",
#    max_completion_tokens=max_completion_tokens,
    temperature=0,
    )

class GeneratedQuestions(BaseModel):
    questions: list[str] = Field(
        description="Questions réalistes qu'un utilisateur pourrait poser à propos de l'ensemble du corpus d'avis clients."
    )

structured_llm = llm.with_structured_output(GeneratedQuestions)

In [123]:
prompt = f"""
Tu travailles sur un RAG permettant d'analyser un corpus d'avis clients concernant des banques en ligne.

Voici un échantillon d'avis clients issu du corpus :

{reviews_text}

À partir de cet échantillon, génère uniquement 20 questions pertinentes qu'un utilisateur pourrait poser à ce RAG, réparties exactement ainsi :
- 14 questions négatives, formulées autour de problèmes, difficultés, reproches ou plaintes ;
- 4 questions positives, formulées autour d'avantages, qualités, bénéfices ou éléments appréciés ;
- 2 questions neutres ou factuelles, formulées autour d'expériences, situations ou faits, sans jugement positif ou négatif.

Contraintes impératives :
- Être formulées de manière naturelle et réaliste.
- Être répondables uniquement à partir des informations présentes dans les avis clients.
- Ne jamais utiliser de questions commençant par « Pourquoi ».
- Ne jamais utiliser le verbe « évaluer » sous quelque forme que ce soit (par exemple, ne jamais utiliser « évaluent-ils »).
- Ne jamais utiliser de formulations impliquant une comparaison, une hiérarchisation, ou une fréquence (« les principaux », « les plus… », « les moins… »).
- Ne jamais mentionner explicitement les banques, les banques en ligne ou les établissements bancaires.
"""

In [124]:
response = structured_llm.invoke(prompt)

In [125]:
response.questions

['Quels types de problèmes rencontrent les clients avec le service client ?',
 "Comment les clients décrivent-ils leur expérience lors de l'ouverture d'un compte ?",
 'Quelles difficultés les utilisateurs rencontrent-ils lors de la validation de leurs opérations ?',
 'Quels reproches sont faits concernant la réactivité du service client ?',
 'Comment les clients se sentent-ils face à des blocages de compte sans explication ?',
 'Quelles plaintes sont formulées concernant les frais bancaires ?',
 'Comment les clients perçoivent-ils la qualité des réponses fournies par le service client ?',
 'Quels problèmes sont signalés concernant la réception de cartes bancaires ?',
 'Comment les clients décrivent-ils leur expérience avec les délais de traitement des demandes ?',
 'Quelles frustrations sont exprimées concernant les communications par e-mail ?',
 'Quels avis négatifs sont partagés sur la gestion des incidents informatiques ?',
 'Comment les clients réagissent-ils face à des erreurs dan

In [126]:
evaluation_path = PROJECT_ROOT / "data/evaluation/evaluation_questions.json"

with open(evaluation_path, "w", encoding="utf-8") as f:
    json.dump(response.questions, f, ensure_ascii=False, indent=2)